# M2 · Prompt Ensemble Bahasa Indonesia
**IF25-40304 Pembelajaran Mesin Multimodal · Kelompok 05**

**Atribusi data:** SEA-VL Indonesian subset (KORIKA-AI/sea-vl_crowdsourcing_id), CC BY-SA 4.0.  
Cahyawijaya et al. (2025), *Crowdsource, Crawl, or Generate? Creating SEA-VL*, ACL.


## 0 · Install dependensi

In [2]:
# Jalankan sekali jika belum terinstall
!pip install sentence-transformers numpy pandas


## 1 · Import dan daftar templat

In [3]:
import json, time, numpy as np

# ── Templat bahasa Indonesia
TEMPLATES_ID = [
    "{}",
    "foto {}",
    "sebuah foto {}",
    "gambar {}",
    "foto budaya Indonesia: {}",
    "dokumentasi budaya Indonesia: {}",
    "{} di Indonesia",
    "sebuah foto yang menunjukkan {}",
    "potret {}",
    "ini adalah foto {}",
]

# ── Kontrol: templat Inggris (memisahkan efek ensemble vs efek bahasa)
TEMPLATES_EN = [
    "{}",
    "a photo of {}",
    "a picture of {}",
    "a photo showing {}",
    "a photo of Indonesian culture: {}",
    "documentation of Indonesian culture: {}",
    "{} in Indonesia",
    "this is a photo of {}",
]

print(f"Templat ID: {len(TEMPLATES_ID)} | Templat EN: {len(TEMPLATES_EN)}")


Templat ID: 10 | Templat EN: 8


## 2 · Fungsi utilitas

In [4]:
def l2norm(x):
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12)

def apply_template(tpl, text, lower_first=True):
    if lower_first and tpl != "{}" and text and text[0].isupper() and not text[:2].isupper():
        text = text[0].lower() + text[1:]
    return tpl.format(text)

def encode_ensemble(encoder, texts, templates, batch_size=64, lower_first=True):
    """Rata-rata embedding teks di bawah tiap templat, lalu normalisasi ulang. -> (n, d)"""
    outs = []
    for tpl in templates:
        wrapped = [apply_template(tpl, t, lower_first) for t in texts]
        outs.append(np.asarray(encoder.encode(wrapped, batch_size=batch_size, normalize_embeddings=True)))
    return l2norm(np.mean(outs, axis=0))

def fuse(e_a, e_b, w):
    """Fusi lambat: w*a + (1-w)*b, dinormalisasi ulang."""
    return l2norm(w * e_a + (1.0 - w) * e_b)

def retrieval_metrics(Q, G, q_group=None, g_group=None, ks=(1, 5, 10), map_k=10):
    nq, ng = len(Q), len(G)
    q_group = np.arange(nq) if q_group is None else np.asarray(q_group)
    g_group = np.arange(ng) if g_group is None else np.asarray(g_group)
    order = np.argsort(-(Q @ G.T), axis=1, kind="stable")
    rel = (g_group[order] == q_group[:, None])
    n_rel = (g_group[None, :] == q_group[:, None]).sum(1)
    out = {f"R@{k}": float(rel[:, :k].any(1).mean()) for k in ks}
    topk = rel[:, :map_k].astype(float)
    prec = np.cumsum(topk, 1) / np.arange(1, map_k + 1)
    out[f"mAP@{map_k}"] = float(((prec * topk).sum(1) / np.minimum(n_rel, map_k)).mean())
    return out

def four_directions(E, Ti, Te, **kw):
    return {
        "gambar->teks ID": retrieval_metrics(E, Ti, **kw),
        "gambar->teks EN": retrieval_metrics(E, Te, **kw),
        "teks ID->gambar": retrieval_metrics(Ti, E, **kw),
        "teks EN->gambar": retrieval_metrics(Te, E, **kw),
    }

print("Fungsi utilitas siap.")


Fungsi utilitas siap.


## 3 · Load cache embedding dari Peran 2

Pastikan file berikut sudah ada di direktori yang benar:
- `hasil_m1/cache/emb_train.npz`
- `hasil_m1/cache/emb_val.npz`

Sesuaikan `CACHE_DIR` jika path berbeda.


In [12]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/MULTIMODAL/Milestone/cache"  # <-- sesuaikan nama shortcut
DATA_DIR  = "/content/drive/MyDrive/MULTIMODAL/bundle-mahasiswa/bundle-mahasiswa"        # <-- folder yang ada train.csv/val.csv

def load(split):
    z = np.load(f"{CACHE_DIR}/emb_{split}.npz")
    return {"img": z["img"], "cap_id": z["id"], "cap_en": z["en"]}

def captions(split):
    df = pd.read_csv(f"{DATA_DIR}/{split}.csv")
    return list(df["caption_id"]), list(df["caption_en"])

tr = load("train")
va = load("val")
print("train  img:", tr["img"].shape, " cap_id:", tr["cap_id"].shape, " cap_en:", tr["cap_en"].shape)
print("val    img:", va["img"].shape, " cap_id:", va["cap_id"].shape, " cap_en:", va["cap_en"].shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train  img: (2072, 512)  cap_id: (2072, 512)  cap_en: (2072, 512)
val    img: (275, 512)  cap_id: (275, 512)  cap_en: (275, 512)


## 4 · Load encoder (sekali saja, butuh koneksi internet pertama kali)

In [8]:
from sentence_transformers import SentenceTransformer

print("Loading encoder gambar + EN...")
img_enc = SentenceTransformer("sentence-transformers/clip-ViT-B-32")
print("Loading encoder teks ID...")
txt_id  = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1")
print("Encoder siap.")


Loading encoder gambar + EN...


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

0_CLIPModel/model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

... (21 outputs left out)

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.57MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

Encoder siap.


## 5 · Seleksi templat (greedy, hanya dari train)

**Anti-leakage**: val.csv tidak disentuh di tahap ini.


In [14]:

SEED   = 0
N_SUB  = 500   # ukuran subset train untuk seleksi (lebih besar = lebih akurat, lebih lambat)
MAX_K  = 5     # maksimum jumlah templat yang dipilih

def greedy_select(encoder, texts_train, img_train, candidates, max_k=5, batch_size=64, tol=0.0):
    per = {t: np.asarray(encoder.encode([apply_template(t, x) for x in texts_train],
                                        batch_size=batch_size, normalize_embeddings=True))
           for t in candidates}
    def score(sel):
        Ti = l2norm(np.mean([per[t] for t in sel], axis=0))
        return (retrieval_metrics(Ti, img_train)["R@1"] +
                retrieval_metrics(img_train, Ti)["R@1"]) / 2
    chosen = ["{}"]
    trace = []
    best = score(chosen)
    trace.append((list(chosen), best))
    while len(chosen) < max_k:
        gains = [(score(chosen + [t]), t) for t in candidates if t not in chosen]
        if not gains: break
        s, t = max(gains)
        if s - best <= tol: break
        chosen.append(t); best = s; trace.append((list(chosen), s))
    return chosen, trace

# Ambil subset train
rng = np.random.default_rng(SEED)
cap_id_train, _ = captions("train")
idx = np.sort(rng.choice(len(cap_id_train), size=min(N_SUB, len(cap_id_train)), replace=False))

t0 = time.time()
chosen, trace = greedy_select(txt_id,
                              [cap_id_train[i] for i in idx],
                              tr["img"][idx],
                              TEMPLATES_ID, max_k=MAX_K)
print(f"Selesai ({time.time()-t0:.0f} dtk)")
print("Templat terpilih:", chosen)
print("\nTrace greedy:")
for s, r in trace:
    print(f"  {s}  →  R@1 train rata-rata dua arah ID = {r:.4f}")

# Simpan
json.dump({"seed": SEED, "n_sub": int(len(idx)), "split": "train", "chosen": chosen,
           "trace": [{"set": s, "R@1_train_rata2_dua_arah_ID": r} for s, r in trace]},
          open("templates_terpilih.json", "w"), ensure_ascii=False, indent=2)
print("\nDisimpan ke templates_terpilih.json")


Selesai (341 dtk)
Templat terpilih: ['{}', 'ini adalah foto {}']

Trace greedy:
  ['{}']  →  R@1 train rata-rata dua arah ID = 0.1580
  ['{}', 'ini adalah foto {}']  →  R@1 train rata-rata dua arah ID = 0.1720

Disimpan ke templates_terpilih.json


## 6 · Evaluasi semua varian di val split




In [15]:
chosen = json.load(open("templates_terpilih.json"))["chosen"]
print("Templat yang digunakan:", chosen)

cap_id_val, cap_en_val = captions("val")

t0 = time.time()
Ti_ens = encode_ensemble(txt_id, cap_id_val, chosen)
Te_ens = encode_ensemble(img_enc, cap_en_val, TEMPLATES_EN[:len(chosen)])  # kontrol EN, jumlah templat sama

results = {
    "V0 baseline M1":        four_directions(va["img"], va["cap_id"], va["cap_en"]),
    "V1 ensemble ID":        four_directions(va["img"], Ti_ens,       va["cap_en"]),
    "K1 kontrol ensemble EN":four_directions(va["img"], va["cap_id"], Te_ens),
}
results["_meta"] = {
    "split": "val", "seed": SEED,
    "templat_ID": chosen,
    "waktu_dtk": round(time.time() - t0, 1)
}

json.dump(results, open("hasil_m2_prompt_ensemble.json", "w"), ensure_ascii=False, indent=2)

# Tampilkan tabel ringkas
print(f"\nWaktu eval: {results['_meta']['waktu_dtk']} dtk\n")
for name, r in results.items():
    if name.startswith("_"): continue
    print(f"{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    for d, m in r.items():
        print(f"  {d:22s}  " + "  ".join(f"{k}={v:.4f}" for k,v in m.items()))
print("\nDisimpan ke hasil_m2_prompt_ensemble.json")


Templat yang digunakan: ['{}', 'ini adalah foto {}']

Waktu eval: 76.8 dtk

  V0 baseline M1
  gambar->teks ID         R@1=0.2145  R@5=0.3745  R@10=0.5055  mAP@10=0.2924
  gambar->teks EN         R@1=0.4436  R@5=0.7127  R@10=0.8327  mAP@10=0.5576
  teks ID->gambar         R@1=0.1964  R@5=0.4182  R@10=0.5055  mAP@10=0.2830
  teks EN->gambar         R@1=0.3818  R@5=0.6727  R@10=0.7709  mAP@10=0.5073
  V1 ensemble ID
  gambar->teks ID         R@1=0.2436  R@5=0.4073  R@10=0.5236  mAP@10=0.3201
  gambar->teks EN         R@1=0.4436  R@5=0.7127  R@10=0.8327  mAP@10=0.5576
  teks ID->gambar         R@1=0.1891  R@5=0.4036  R@10=0.5200  mAP@10=0.2803
  teks EN->gambar         R@1=0.3818  R@5=0.6727  R@10=0.7709  mAP@10=0.5073
  K1 kontrol ensemble EN
  gambar->teks ID         R@1=0.2145  R@5=0.3745  R@10=0.5055  mAP@10=0.2924
  gambar->teks EN         R@1=0.4509  R@5=0.7455  R@10=0.8436  mAP@10=0.5713
  teks ID->gambar         R@1=0.1964  R@5=0.4182  R@10=0.5055  mAP@10=0.2830
  teks EN->gambar 

## 7 · Ringkasan tabel untuk laporan

In [16]:
res = json.load(open("hasil_m2_prompt_ensemble.json"))
arah_map = {
    "teks ID->gambar":  "ID→G",
    "gambar->teks ID":  "G→ID",
    "teks EN->gambar":  "EN→G",
    "gambar->teks EN":  "G→EN",
}
variants = [k for k in res if not k.startswith("_")]
header = ["Varian"] + [f"{a} R@1" for a in arah_map.values()] + ["mAP@10 ID→G"]
print("\t".join(header))
for v in variants:
    row = [v]
    for arah_key in arah_map:
        row.append(f"{res[v][arah_key]['R@1']:.4f}")
    row.append(f"{res[v]['teks ID->gambar']['mAP@10']:.4f}")
    print("\t".join(row))


Varian	ID→G R@1	G→ID R@1	EN→G R@1	G→EN R@1	mAP@10 ID→G
V0 baseline M1	0.1964	0.2145	0.3818	0.4436	0.2830
V1 ensemble ID	0.1891	0.2436	0.3818	0.4436	0.2803
K1 kontrol ensemble EN	0.1964	0.2145	0.3818	0.4509	0.2830


## 8 · Selftest logika (opsional, tanpa model/data)

In [17]:
import zlib

class _FakeEnc:
    def __init__(self, d=64): self.d = d
    def _w(self, w):
        return np.random.default_rng(zlib.crc32(w.encode())).normal(size=self.d)
    def encode(self, texts, batch_size=0, normalize_embeddings=True):
        return l2norm(np.stack([sum(self._w(w) for w in t.lower().split()) for t in texts]))

enc, N = _FakeEnc(), 200
caps = [f"penari {i} memakai kostum {i % 7} di panggung {i % 11}" for i in range(N)]
img  = enc.encode(caps)

m = retrieval_metrics(img, img)
assert m["R@1"] == 1.0 and abs(m["mAP@10"] - 1.0) < 1e-9
rnd = l2norm(np.random.default_rng(0).normal(size=(N, 64)))
assert retrieval_metrics(rnd, img)["R@1"] < 0.05
e1 = encode_ensemble(enc, caps, ["{}"]); assert np.allclose(e1, img)
e2 = encode_ensemble(enc, caps, TEMPLATES_ID)
assert e2.shape == img.shape and np.allclose(np.linalg.norm(e2, axis=1), 1)
ch, _ = greedy_select(enc, caps, img, TEMPLATES_ID, max_k=3)
assert ch[0] == "{}" and len(ch) <= 3

print("selftest OK", ch)


selftest OK | templat terpilih (encoder tiruan, tidak bermakna): ['{}']
